# Guideline 36 (g36) Extension Classes

ASHRAE Guideline 36 defines standardized HVAC control sequences of operation. The
223P ontology ships a `g36:` extension namespace
(`http://data.ashrae.org/standard223/1.0/extensions/g36#`) that layers extra
constraints on top of core `s223:` classes - e.g. "a G36 `Zone` must expose a CO2
property, an occupancy property, and a window switch, each via `hasProperty`" -
so that a model can be checked for Guideline-36-readiness, not just 223P-validity.

It's vendored in the same file as the base ontology,
`src/semantic_objects/ontologies/s223/223p.ttl`, just under the `g36:` prefix instead
of `s223:`, and generated as typed Python the same way `semantic_objects.s223` is:

```
python -m semantic_objects.ingest.cli --ontology g36
```

through the exact same ontology-agnostic parser/SHACL-classifier/emitter pipeline as
`s223`, using a small `G36Adapter` (`src/semantic_objects/ingest/adapters/g36.py`).
Because g36 classes subclass `s223:` classes and reuse `s223:` relations
(`hasProperty`, `hasDomain`, `connectedTo`, ...) rather than defining their own,
`G36Adapter` points those cross-namespace references back at the already-generated
`semantic_objects.s223` classes/relations instead of redefining them.

This notebook is about *using* the result - see `ontology-ingestion-tutorial.ipynb`
for how the pipeline and the g36 adapter specifically work (including the raw SHACL
shapes behind these classes), and `s223-generated-classes-tutorial.ipynb` for the
same everyday-use tour of `semantic_objects.s223`. Section 1 looks at `g36.Fan`/
`g36.FanWithVFD` and `g36.Zone`; section 2 builds a hand-written property extension
(`ThresholdAlarm`) on top of what's generated and attaches it to a real `g36.Zone` -
the same generated-base/hand-written-extension pattern
`semantic_objects.s223.properties` uses for `QuantifiableObservableProperty`.


## 1. The generated `semantic_objects.g36` classes

`entities.py`/`properties.py`/`enumerationkinds.py`/`relations.py` under
`semantic_objects.g36` mirror the `semantic_objects.s223` layout exactly (a
hand-written module doing `from ._generated.X import *`, generated content
underneath, regenerated content never hand-edited) - `g36` just happens to have
nothing of its own in the properties/enumerationkinds/relations buckets, since every
g36 class is an entity that reuses s223's Property hierarchy and relations rather
than defining new ones.


In [1]:
from semantic_objects import g36
from semantic_objects.s223 import entities as s223_entities, properties as s223_properties, relations as s223_relations

print(g36.Zone.__bases__, "-", g36.Zone.comment)
print(g36.Fan.__bases__, "-", g36.Fan.comment)
print()
print("g36.Zone really does subclass the generated s223 Zone:", issubclass(g36.Zone, s223_entities.Zone))
print("g36.relations reuses the same Predicate objects as s223.relations:",
      g36._generated.relations.hasProperty is s223_relations.hasProperty)


CRITICAL:root:Install the 'bacnet-ingress' module, e.g. 'pip install buildingmotif[bacnet-ingress]'


(<class 'semantic_objects.s223._generated.entities.Zone'>,) - A thermal zone with the points required for Guideline 36 control sequences. It is a collection of s223:DomainSpace instances.
(<class 'semantic_objects.s223._generated.entities.Fan'>,) - A fan with a start/stop command.

g36.Zone really does subclass the generated s223 Zone: True
g36.relations reuses the same Predicate objects as s223.relations: True


### 1.1 Fans: `Fan` and `FanWithVFD`

`g36:Fan` adds one required start/stop command to `s223:Fan`; `g36:FanWithVFD`
subclasses `g36:Fan` itself and layers on a second one (a speed command) - fields
accumulate up the g36 hierarchy exactly the way they do for any other generated
class hierarchy.


In [2]:
for cls in (g36.Fan, g36.FanWithVFD):
    print(f"--- {cls.__name__} ({cls.comment}) ---")
    for name, f in cls.__dataclass_fields__.items():
        print(f"  {name}: {f.type.__name__}  (relation={f.metadata['relation'].__name__})")
    print()


--- Fan (A fan with a start/stop command.) ---
  outlet_connection_point: OutletConnectionPoint  (relation=hasConnectionPoint)
  inlet_connection_point: InletConnectionPoint  (relation=hasConnectionPoint)
  enumerated_actuatable_property: EnumeratedActuatableProperty  (relation=hasProperty)

--- FanWithVFD (A fan controlled by a VFD.) ---
  outlet_connection_point: OutletConnectionPoint  (relation=hasConnectionPoint)
  inlet_connection_point: InletConnectionPoint  (relation=hasConnectionPoint)
  enumerated_actuatable_property: EnumeratedActuatableProperty  (relation=hasProperty)
  quantifiable_actuatable_property: QuantifiableActuatableProperty  (relation=hasProperty)



In [3]:
print(g36.Fan.to_yaml())

Fan:
  body: >+
    @prefix P: <urn:___param___#> .

    @prefix s223: <http://data.ashrae.org/standard223#> .


    P:name a s223:Fan ;
        s223:hasConnectionPoint P:inlet_connection_point,
            P:outlet_connection_point ;
        s223:hasProperty P:enumerated_actuatable_property .

  dependencies:
  - args:
      name: outlet_connection_point
    template: OutletConnectionPoint
  - args:
      name: inlet_connection_point
    template: InletConnectionPoint
  - args:
      name: enumerated_actuatable_property
    template: EnumeratedActuatableProperty



Both classes' SHACL also carries a constraint that isn't representable as a single
typed field: the required `hasProperty` target must itself satisfy a nested
constraint - `s223:hasEnumerationKind a s223:Binary-OnOff` for `Fan`,
`qudt:hasQuantityKind = quantitykind:DimensionlessRatio` for `FanWithVFD` - narrowing
*which* property satisfies the field, not just its class. These are preserved as
supplementary notes on the field rather than enforced:


In [4]:
from semantic_objects.g36._generated import _raw_shapes

for cls_name in ('Fan', 'FanWithVFD'):
    print(f"--- {cls_name} ---")
    for entry in _raw_shapes.RAW_SHAPES[cls_name]:
        print(f"  [{entry['kind']}] field={entry.get('field_name')}: {entry.get('message') or entry.get('comment')}")


--- Fan ---
  [nested-node] field=enumerated_actuatable_property: g36: A Fan shall have at least one Start/Stop command using the relation hasProperty.
--- FanWithVFD ---
  [nested-node] field=quantifiable_actuatable_property: g36: A fan with VFD shall have at least one fan speed command using the relation hasProperty.


Instantiating both shows the fields accumulating - `FanWithVFD` needs the connection
points *and* both properties. It also shows something g36-specific: per the
ontology's own design, no instance is ever *asserted* as `g36:Fan`/`g36:FanWithVFD`
directly - each g36 class has a paired `g36:XAnnotation` SHACL rule that infers the
type once an instance's own triples satisfy its constraints (see the vendored
`223p.ttl` for `g36:FanAnnotation`, or `ontology-ingestion-tutorial.ipynb` section 7
for how the generator accounts for this). So `FanWithVFD.get_sparql_query()` below
looks for `a s223:Fan` - the real ontology term it specializes - not a fictitious
`s223:FanWithVFD`, because the generator resolves each g36 class's `_name` to its
nearest real s223 ancestor rather than defaulting to the class's own Python name.


In [5]:
from semantic_objects.s223 import enumerationkinds

def build_fan(cls, name, **extra):
    air = enumerationkinds.Air()
    connection = s223_entities.Connection(medium=air)
    outlet = s223_entities.OutletConnectionPoint(medium=air, connection=connection)
    inlet = s223_entities.InletConnectionPoint(medium=air, connection=connection)
    start_stop = s223_properties.EnumeratedActuatableProperty(enumeration_kind=enumerationkinds.OnOff())
    obj = cls(outlet_connection_point=outlet, inlet_connection_point=inlet,
              enumerated_actuatable_property=start_stop, **extra)
    obj._name = name
    return obj

fan = build_fan(g36.Fan, "SupplyFan_1")
fan_vfd = build_fan(g36.FanWithVFD, "SupplyFan_2",
                     quantifiable_actuatable_property=s223_properties.QuantifiableActuatableProperty())

print(fan._name, "->", type(fan)._name)
print(fan_vfd._name, "->", type(fan_vfd)._name)
print()

query = g36.FanWithVFD.get_sparql_query()
print("s223:Fan" in query, "s223:FanWithVFD" in query)
print(query)


SupplyFan_1 -> Fan
SupplyFan_2 -> Fan

True False
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX s223: <http://data.ashrae.org/standard223#>
SELECT DISTINCT * WHERE { ?name s223:hasProperty ?enumerated_actuatable_property .
?name s223:hasProperty ?quantifiable_actuatable_property .
?inlet_connection_point rdf:type s223:InletConnectionPoint .
?outlet_connection_point rdf:type s223:OutletConnectionPoint .
?enumerated_actuatable_property rdf:type s223:EnumeratedActuatableProperty .
?quantifiable_actuatable_property rdf:type s223:QuantifiableActuatableProperty .
?name rdf:type s223:Fan .
?name s223:hasConnectionPoint ?inlet_connection_point .
?name s223:hasConnectionPoint ?outlet_connection_point . }


`generate_rdf_class_definition()` re-derives a SHACL shape from the Python class -
by default only the constraints declared *on that class*, which for `FanWithVFD` is
just the one new speed-command field, not the connection points/start-stop command
it inherits from `Fan`. Pass `include_hierarchy=True` to get the full accumulated
shape instead (it also picks up the optional relations `FanWithVFD` inherits from
`Equipment`/`Connectable`, like `hasPhysicalLocation`) - useful whenever you need
something that actually validates a `FanWithVFD` on its own, not just its
incremental difference from `Fan`.


In [6]:
print("--- FanWithVFD, own constraints only ---")
print(g36.FanWithVFD.generate_rdf_class_definition())
print("--- FanWithVFD, include_hierarchy=True ---")
print(g36.FanWithVFD.generate_rdf_class_definition(include_hierarchy=True))


--- FanWithVFD, own constraints only ---
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

s223:Fan a s223:Class,
        rdfs:Class,
        sh:NodeShape ;
    rdfs:label "FanWithVFD" ;
    rdfs:comment "A fan controlled by a VFD." ;
    rdfs:subClassOf s223:Equipment ;
    sh:property [ a sh:PropertyShape ;
            rdfs:comment "If the relation `hasProperty` is present it must associate the `FanWithVFD` with a `QuantifiableActuatableProperty`." ;
            sh:class s223:QuantifiableActuatableProperty ;
            sh:message "s223: If the relation `hasProperty` is present it must associate the `FanWithVFD` with a `QuantifiableActuatableProperty`." ;
            sh:minCount 1 ;
            sh:path s223:hasProperty ;
            sh:qualifiedMinCount 1 ;
            sh:qualifiedValueShape [ a sh:NodeShape ;
                    

### 1.2 Zones

`g36:Zone` adds four `sh:Info`-severity recommended properties (temperature setpoint
adjustment, CO2 concentration, occupancy, window switch) plus one hard
`sh:Violation`-severity requirement (`hasDomain = s223:Domain-HVAC`) on top of
`s223:Zone`. The four recommended properties became real fields (both
`QuantifiableObservableProperty` shapes collapse to fields with the same base
name - `quantifiable_observable_property`/`_2` - since disambiguating them further
would need the nested aspect/quantity-kind constraints, which are only preserved as
notes, not enforced); the domain requirement and the window switch's
alternative-path constraint aren't representable as a single typed field, so they
land in `_raw_shapes.RAW_SHAPES` instead.


In [7]:
print(g36.Zone.comment)
print()
for name, f in g36.Zone.__dataclass_fields__.items():
    print(f"{name}: {f.type.__name__}  (relation={f.metadata['relation'].__name__})")
print()
for entry in _raw_shapes.RAW_SHAPES['Zone']:
    print(f"[{entry['kind']}] field={entry.get('field_name')}: {entry.get('message') or entry.get('comment')}")


A thermal zone with the points required for Guideline 36 control sequences. It is a collection of s223:DomainSpace instances.

domain: Domain  (relation=hasDomain)
quantifiable_observable_property: QuantifiableObservableProperty  (relation=hasProperty)
quantifiable_observable_property_2: QuantifiableObservableProperty  (relation=hasProperty)
enumerated_observable_property: EnumeratedObservableProperty  (relation=hasProperty)
enumerated_observable_property_2: EnumeratedObservableProperty  (relation=hasProperty)

[nested-node] field=quantifiable_observable_property: g36: A Zone shall have a zone temperature setpoint adjustment property using the relation hasProperty, if applicable.
[nested-node] field=quantifiable_observable_property_2: g36: A Zone shall have at least a zone CO2 concentration property using the relation hasProperty, if applicable control is used.
[nested-node] field=enumerated_observable_property: g36: A Zone shall have at least one binary zone occupancy property using t

In [8]:
print(g36.Zone.get_sparql_query())
print()
print(g36.Zone.generate_rdf_class_definition(include_hierarchy=True))


PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX s223: <http://data.ashrae.org/standard223#>
SELECT DISTINCT * WHERE { ?name s223:hasProperty ?quantifiable_observable_property_2 .
?quantifiable_observable_property_2 rdf:type s223:QuantifiableObservableProperty .
?enumerated_observable_property rdf:type s223:EnumeratedObservableProperty .
?domain rdf:type s223:EnumerationKind-Domain .
?name rdf:type s223:Zone .
?name s223:hasDomain ?domain .
?quantifiable_observable_property rdf:type s223:QuantifiableObservableProperty .
?name s223:hasProperty ?quantifiable_observable_property .
?name s223:hasProperty ?enumerated_observable_property_2 .
?name s223:hasProperty ?enumerated_observable_property .
?enumerated_observable_property_2 rdf:type s223:EnumeratedObservableProperty . }

@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#

## 2. A new property: `ThresholdAlarm`

Section 8 of `s223-generated-classes-tutorial.ipynb` covers the extension pattern:
`semantic_objects.s223.properties.QuantifiableObservableProperty` is a hand-written
subclass of the *generated* base of the same name, adding `qk`/`value`/`unit` fields
the ingestion pipeline can't derive (they live in the QUDT namespace, out of scope
for this pilot). Concrete leaf classes like `Area`/`Power`/`Tilt` build on that
extension; `Area_SP` shows how to pin a property to a fixed set of `s223:Aspect-*`
values using `exact_values` field metadata (see
`docs/exact_values_generalization.md`) rather than a single fixed value, since a
setpoint-flavored `Area` can carry more than one aspect at once.

`ThresholdAlarm` follows the same `Area_SP` pattern: any
`QuantifiableObservableProperty` (so still generic over quantity kind - a duct
static-pressure alarm, a CO2 high-limit alarm, ...) that's flagged with *both*
`s223:Aspect-Threshold` and `s223:Aspect-Alarm`, i.e. "this reading crossing its
threshold is itself the alarm condition." We define it in the notebook rather than
editing `properties.py`, but the pattern is identical either way: subclass the
existing generated/hand-written base, never edit `_generated/*.py` directly.


In [9]:
from typing import Optional
from dataclasses import field as dc_field

from semantic_objects.core import semantic_object
from semantic_objects.s223 import properties, enumerationkinds
from semantic_objects.s223.relations import hasAspect
from semantic_objects.qudt import quantitykinds


@semantic_object
class ThresholdAlarm(properties.QuantifiableObservableProperty):
    """A QuantifiableObservableProperty that is both a threshold and an alarm -
    e.g. a duct static-pressure high-limit alarm, a CO2 high-limit alarm."""
    _semantic_type = properties.QuantifiableObservableProperty
    aspects: Optional[list] = dc_field(
        default=None,
        init=False,
        metadata={
            'relation': hasAspect,
            'exact_values': [enumerationkinds.Threshold, enumerationkinds.Alarm],
            'qualified': False,
        },
    )

print("ThresholdAlarm defined")


ThresholdAlarm defined


In [10]:
for name, f in ThresholdAlarm.__dataclass_fields__.items():
    print(f"{name}: {f.type}  (relation={f.metadata['relation'].__name__}, "
          f"exact_values={[v.__name__ for v in f.metadata['exact_values']] if f.metadata.get('exact_values') else None})")


qk: <class 'semantic_objects.qudt.quantitykinds.QuantityKind'>  (relation=hasQuantityKind, exact_values=None)
value: <class 'float'>  (relation=hasValue, exact_values=None)
unit: typing.Optional[semantic_objects.units.Unit]  (relation=hasUnit, exact_values=None)
aspects: typing.Optional[list]  (relation=hasAspect, exact_values=['Threshold', 'Alarm'])


In [11]:
alarm = ThresholdAlarm(qk=quantitykinds.Pressure, value=250.0)
alarm._name = "DuctStaticPressure_HighAlarm"
print(alarm._name, "->", alarm.value, alarm.unit._name, alarm.qk._name)


DuctStaticPressure_HighAlarm -> 250.0 PA Pressure


### 2.1 Attaching it to an already-ingested `g36.Zone`

`ThresholdAlarm` *is a* `QuantifiableObservableProperty`, and `g36.Zone`'s
`quantifiable_observable_property`/`quantifiable_observable_property_2` fields (see
section 1.2) both accept any instance of that class - so a real `ThresholdAlarm`
slots directly into a real, generated `g36.Zone`, no different from any other
`QuantifiableObservableProperty` value. Nothing here is g36-specific or notebook-only
generated code; the only hand-written class is `ThresholdAlarm` itself.


In [12]:
zone = g36.Zone(
    domain=enumerationkinds.HVAC(),
    quantifiable_observable_property=alarm,
    quantifiable_observable_property_2=properties.QuantifiableObservableProperty(qk=quantitykinds.Pressure, value=200.0),
    enumerated_observable_property=properties.EnumeratedObservableProperty(enumeration_kind=enumerationkinds.Occupancy()),
    enumerated_observable_property_2=properties.EnumeratedObservableProperty(enumeration_kind=enumerationkinds.OnOff()),
)
zone._name = "Zone_204"

held = zone.quantifiable_observable_property
print(f"{zone._name}.quantifiable_observable_property -> {held._name} "
      f"(a real ThresholdAlarm: {isinstance(held, ThresholdAlarm)})")


Zone_204.quantifiable_observable_property -> DuctStaticPressure_HighAlarm (a real ThresholdAlarm: True)


`get_sparql_query()` turns the `exact_values` metadata into a `FILTER ... IN (...)`
clause, so a query for `ThresholdAlarm` only matches properties carrying *both*
aspects - not properties that happen to carry just one of them:


In [13]:
print(ThresholdAlarm.get_sparql_query())


PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX quantitykind: <http://qudt.org/vocab/quantitykind/>
PREFIX s223: <http://data.ashrae.org/standard223#>
SELECT DISTINCT * WHERE {  ?name rdf:type s223:ThresholdAlarm .
?qk rdf:type quantitykind:QuantityKind .
?name s223:hasQuantityKind ?qk .
?name s223:hasValue ?value .
?name <http://data.ashrae.org/standard223#hasAspect> ?name_exact_values .
FILTER(?name_exact_values IN (s223:Aspect-Threshold,s223:Aspect-Alarm) )  }


In [14]:
print(ThresholdAlarm.to_yaml())


ThresholdAlarm:
  body: >+
    @prefix P: <urn:___param___#> .

    @prefix s223: <http://data.ashrae.org/standard223#> .


    P:name a s223:QuantifiableObservableProperty ;
        s223:hasAspect P:aspects ;
        s223:hasQuantityKind P:qk ;
        s223:hasUnit P:unit ;
        s223:hasValue P:value .

  dependencies: []



## Next steps

- `ontology-ingestion-tutorial.ipynb` - how the shared generation pipeline works in
  general (parser, SHACL classifier, emitter), and (section 7) the g36-specific
  adapter: scoping the walk to `g36:`, pointing cross-namespace parents/relations
  back at `semantic_objects.s223`, and forcing `_name` to the real s223 ancestor so
  instances are never asserted under a fictitious `s223:<G36ClassName>` term.
- `src/semantic_objects/ingest/adapters/g36.py` - the adapter itself.
- `docs/exact_values_generalization.md` - the `exact_values` mechanism used for
  `ThresholdAlarm.aspects` above.
- `src/semantic_objects/g36/properties.py` - where `ThresholdAlarm` would live if
  promoted from notebook experiment to a real reusable class.
- `s223-generated-classes-tutorial.ipynb` - the same tour (fields, SPARQL, SHACL,
  BuildingMOTIF templates) for `semantic_objects.s223`, in more depth.
